# Context Sensitivity Analysis

Analyze how model behavior depends on token position, context length,
local vs. distant information, and in-context learning dynamics.

This notebook covers the `irtk.context_sensitivity` module.

## Why This Matters

Context sensitivity analysis measures how a position's representation changes as the surrounding context varies. This reveals whether heads operate locally (nearby tokens only) or globally, how the effective context window compares to the architectural maximum, and how in-context learning unfolds.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import context_sensitivity

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Positional Attention Profile

How does attention weight vary with token distance?

In [ ]:
prompts = [
    "The Eiffel Tower is located in Paris France",
    "The capital of France is Paris which is",
    "London is the capital of England and it",
]
token_seqs = [model.to_tokens(p) for p in prompts]

result = context_sensitivity.positional_attention_profile(model, token_seqs)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, (l, h) in enumerate([(0, 0), (0, 7), (6, 0), (11, 0)]):
    ax = axes[i // 2, i % 2]
    profile = result['profiles'][l, h, :15]  # first 15 distances
    ax.bar(range(len(profile)), profile, color='steelblue')
    ax.set_xlabel('Distance (tokens back)')
    ax.set_ylabel('Mean Attention')
    ax.set_title(f'L{l}H{h} (dominant dist: {result["dominant_distances"][l,h]})')

plt.tight_layout()
plt.show()

## 2. Local vs Global Score

Which heads attend locally vs. globally?

In [ ]:
result = context_sensitivity.local_vs_global_score(model, token_seqs, local_window=3)

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(result['local_fractions'], cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xlabel('Head')
ax.set_ylabel('Layer')
ax.set_title('Local Attention Fraction (window=3)')
plt.colorbar(im, ax=ax, label='Local Fraction')
plt.tight_layout()
plt.show()

print("Most local heads:")
for l, h in result['most_local_heads'][:5]:
    print(f"  L{l}H{h}: local = {result['local_fractions'][l,h]:.2%}")

print("\nMost global heads:")
for l, h in result['most_global_heads'][:5]:
    print(f"  L{l}H{h}: global = {result['global_fractions'][l,h]:.2%}")

## 3. Context Length Sensitivity

How do predictions change as we add more context?

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")
paris_id = tokenizer.encode(" Paris")[0]
def metric(logits):
    return float(logits[-1, paris_id])

result = context_sensitivity.context_length_sensitivity(model, tokens, metric)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(result['lengths'], result['metrics'], 'bo-')
ax.axvline(result['convergence_length'], color='green', alpha=0.3,
           label=f'Convergence: {result["convergence_length"]}')
ax.axvline(result['max_change_length'], color='red', alpha=0.3,
           label=f'Max change: {result["max_change_length"]}')
ax.set_xlabel('Context Length (tokens)')
ax.set_ylabel('Paris Logit')
ax.set_title('Prediction vs Context Length')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Token Distance Effect

Which positions at which distances matter most for the prediction?

In [ ]:
result = context_sensitivity.token_distance_effect(model, tokens, -1, metric)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(result['distances'], np.abs(result['effects']), color='steelblue')
ax.axvline(result['effective_window'], color='red', linestyle='--',
           label=f'Effective window: {result["effective_window"]}')
ax.set_xlabel('Distance from Target Position')
ax.set_ylabel('|Effect on Metric|')
ax.set_title(f'Token Distance Effect (peak at distance {result["peak_distance"]})')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `positional_attention_profile()` | Attention weight vs. token distance |
| `local_vs_global_score()` | Classify heads as local vs. global |
| `context_length_sensitivity()` | How predictions change with context length |
| `in_context_learning_dynamics()` | Track representations as ICL examples are added |
| `token_distance_effect()` | Effective receptive field via ablation |